# 00 · 환경과 COST2100 데이터 확인

이 노트북은 경로와 **validation split**만 확인합니다. 모델 학습과 test 평가는 실행하지 않습니다. Colab에서는 Drive에 둔 프로젝트 폴더를 사용하고 데이터 파일을 세션 디스크로 복사합니다.

In [ ]:
from pathlib import Path
import os, shutil, sys
import numpy as np
import torch

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

# Drive 폴더 위치가 다르면 이 변수를 직접 지정하세요.
PROJECT_ROOT_OVERRIDE = None  # 예: '/content/drive/MyDrive/연구/CSI_SLLM_RESEARCH'
if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
DATA_DIR = PROJECT_ROOT / 'Data'
assert DATA_DIR.is_dir(), f'Data 폴더를 찾을 수 없습니다: {DATA_DIR}'

if IN_COLAB:
    ACTIVE_DATA_DIR = Path('/content/cost2100_indoor')
    ACTIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name in ['DATA_Hvalin.mat']:
        src, dst = DATA_DIR / name, ACTIVE_DATA_DIR / name
        assert src.is_file(), f'필수 데이터 파일 없음: {src}'
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
else:
    ACTIVE_DATA_DIR = DATA_DIR

sys.path.insert(0, str(PROJECT_ROOT))
from src.cost2100 import COST2100Dataset
print('Project:', PROJECT_ROOT)
print('Data:', ACTIVE_DATA_DIR)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
val = COST2100Dataset(ACTIVE_DATA_DIR / 'DATA_Hvalin.mat')
x = val.data
print('Validation samples:', len(val))
print('Tensor shape:', tuple(x.shape))
print('dtype:', x.dtype, '| finite:', bool(torch.isfinite(x).all()))
print('range:', float(x.min()), 'to', float(x.max()))
assert tuple(x.shape[1:]) == (2, 32, 32)
assert torch.isfinite(x).all()
assert float(x.min()) >= 0 and float(x.max()) <= 1

## 진행 조건

출력에서 validation 샘플 수가 30,000, 모양이 `(30000, 2, 32, 32)`인지 확인하세요. 경로/형태 assertion이 실패하면 학습을 진행하지 말고 `DATA_DIR` 또는 MAT 파일을 확인합니다. Colab에서는 Drive 폴더를 통째로 올리는 대신 최소 세 split만 세션 저장소로 복사합니다.